# logit-thermo A2-7：滴定のデータ集め（Pythia-1B、GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。見込みは約 1 時間。終わったら自動で切れます。
- ゲート：`docs/A2_7_GATE.md`。Pythia-1B で、禁止する写しの長さ n ＝ 8・6・3・2 の 4 回（温度は T_0 のまわりの 8 つ）。A2-7 のデータ集めはモデルごとに 3 本あり、同時に回して大丈夫です。
- 計算のコード：`scripts/a2_2_collect.py`・`scripts/a2_4_collect.py` は Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_7_collect.py`（`8f8776cbe4a0e1a7…`）は埋め込み。
- 保存先 `A2_7/`。途中で切れても、もう一度「すべて実行」すれば終わった回は飛ばします。
- **通知**：🚀・回ごとの ✅・🎉・⚠️。名札は「logit-thermo A2-7 滴定 データ集め（Pythia-1B、n ＝ 8・6・3・2、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-7 滴定 データ集め（Pythia-1B、n ＝ 8・6・3・2、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-7 titration pythia', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRC24': '6ef87866ae7d895d63c6e07d48a58a2ddf4111a4802ffe98dbe7f61892c96b57'})
SRC22, SRC24 = _s['SRC22'], _s['SRC24']
SRC7C = '"""A2-7（ゲート docs/A2_7_GATE.md）のデータ集め：滴定（写しの禁止の濃さを変える）。GPU（L4）のノートブックで、\nscripts/a2_2_collect.py と scripts/a2_4_collect.py を実行したのと同じ名前空間で実行する（step_stats・split_doc・load_model・\nvalid_mask・h2_of・SEED4・TEMPS・CONT・SAMPLES を変えずに使う）。\n\n濃さ ＝ 禁止する写しの長さ n（n トークンの完全な写しを禁止する ＝ no_repeat_ngram_size ＝ n）。n が小さいほど濃い。\nA2-4 の block は n ＝ 4。ここでは n ＝ 8・6・3・2 を新しく回す（なし と n ＝ 4 は A2-4 のデータを使う）。\n温度は A7_TEMPS の 8 つだけ（T_0 のまわり）。生成の種は A2-4 と同じく SEED4 ＋ 1000 × k ＋ s（k は A2-2 の 15 の温度での番号）。\n"""\nimport gc, os, time\nimport numpy as np\nimport torch\n\nA7_NS = [8, 6, 3, 2]\nA7_TEMPS = [0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05]\nA7_GROUPS = {\'pythia\': (\'pythia-1b\', \'EleutherAI/pythia-1b\'), \'olmo\': (\'olmo2-1b\', \'allenai/OLMo-2-0425-1B\'),\n             \'qwen\': (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\')}\n\n\nclass CopyIndexN:\n    """1 本の文の「直前の n − 1 トークン → それまでに続いたトークンの集合」。n ＝ 4 で A2-4 の CopyIndex と同じ。"""\n\n    def __init__(self, hist, n):\n        self.n, self.seq, self.nxt = n, [int(t) for t in hist], {}\n        m = n - 1\n        for i in range(m, len(self.seq)):\n            self.nxt.setdefault(tuple(self.seq[i - m:i]), set()).add(self.seq[i])\n\n    def candidates(self):\n        m = self.n - 1\n        if len(self.seq) < m:\n            return ()\n        return tuple(self.nxt.get(tuple(self.seq[-m:]), ()))\n\n    def push(self, t):\n        t, m = int(t), self.n - 1\n        if len(self.seq) >= m:\n            self.nxt.setdefault(tuple(self.seq[-m:]), set()).add(t)\n        self.seq.append(t)\n\n\n@torch.no_grad()\ndef generate_ngram(model, prompts, T, valid, gen, H2, n):\n    """A2-4 の generate_pen の \'block\' と同じ手順で、禁止する写しの長さだけ n にしたもの。"""\n    device = valid.device\n    B = len(prompts)\n    x = torch.tensor(prompts, device=device)\n    out = model(x, use_cache=True)\n    past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    idx = [CopyIndexN(p, n) for p in prompts]\n    rec = {k: [] for k in [\'ell\', \'top1\', \'ent\', \'ell2\', \'cand\', \'banned_top1\', \'ban_mass\', \'toks\']}\n    for t in range(CONT):\n        ell, top1, ent, ell2 = step_stats(logits, T, valid, H2)          # つまみ前の logit（本命）\n        z = logits.double().masked_fill(~valid, float(\'-inf\'))\n        lp = torch.log_softmax(z / T, dim=-1)\n        argmax = z.argmax(-1)\n        cands = [idx[b].candidates() for b in range(B)]\n        cand = torch.tensor([len(c) > 0 for c in cands], device=device)\n        rows, cols = [], []\n        for b, c in enumerate(cands):\n            if c:\n                rows += [b] * len(c); cols += list(c)\n        banned_top1 = torch.zeros(B, dtype=torch.bool, device=device)\n        ban_mass = torch.zeros(B, dtype=torch.float64, device=device)\n        zp = z.clone()\n        if rows:\n            r = torch.tensor(rows, device=device); c = torch.tensor(cols, device=device)\n            zp[r, c] = float(\'-inf\')\n            banned_top1[r[c == argmax[r]]] = True\n            ban_mass.index_add_(0, r, torch.exp(lp[r, c]))\n        p = torch.softmax(zp / T, dim=-1)\n        nxt = torch.multinomial(p, 1, generator=gen)\n        for b in range(B):\n            idx[b].push(int(nxt[b, 0]))\n        for k, v in [(\'ell\', ell), (\'top1\', top1), (\'ent\', ent), (\'ell2\', ell2), (\'cand\', cand), (\'banned_top1\', banned_top1),\n                     (\'ban_mass\', ban_mass), (\'toks\', nxt[:, 0])]:\n            rec[k].append(v)\n        out = model(nxt, past_key_values=past, use_cache=True)\n        past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    res = {}\n    for k, v in rec.items():\n        a = torch.stack(v, 1)\n        res[k] = a.cpu().numpy().astype(np.int32 if k == \'toks\' else (np.bool_ if a.dtype == torch.bool else np.float32))\n    return res\n\n\ndef run_ngram(name, hf_id, docs, outdir, device, n, notify=print):\n    path = f\'{outdir}/{name}@n{n}.npz\'\n    if os.path.exists(path):\n        print(name, n, \'はもう終わっています（飛ばします）\')\n        return\n    t0 = time.time()\n    tok, model = load_model(hf_id, device)\n    assert next(model.parameters()).dtype == torch.float32\n    valid, eos = valid_mask(tok, model, device)\n    H2 = h2_of(len(tok))\n    pairs = [split_doc(tok, d[\'text\']) for d in docs]\n    use = [i for i, p in enumerate(pairs) if p is not None]\n    nD = len(use)\n    plen = {len(pairs[i][0]) for i in use}\n    assert len(plen) == 1, plen\n    prompts = [pairs[i][0] for i in use]\n    humans = [pairs[i][1] for i in use]\n    ks = [TEMPS.index(T) for T in A7_TEMPS]\n    keys_f, keys_b = [\'ell\', \'top1\', \'ent\', \'ell2\', \'ban_mass\'], [\'cand\', \'banned_top1\']\n    G = {k: np.zeros((len(ks), SAMPLES, nD, CONT), np.float32) for k in keys_f}\n    G.update({k: np.zeros((len(ks), SAMPLES, nD, CONT), np.bool_) for k in keys_b})\n    toks = np.zeros((len(ks), SAMPLES, nD, CONT), np.int32)\n    for q, k in enumerate(ks):\n        T = TEMPS[k]\n        for s in range(SAMPLES):\n            g = torch.Generator(device=device).manual_seed(SEED4 + 1000 * k + s)\n            r = generate_ngram(model, prompts, T, valid, g, H2, n)\n            for kk in keys_f + keys_b:\n                G[kk][q, s] = r[kk]\n            toks[q, s] = r[\'toks\']\n        print(f\'  {name}@n{n} T={T} おわり（{time.time() - t0:.0f} 秒）\')\n    np.savez_compressed(path + \'.part.npz\', toks=toks, temps=np.array(A7_TEMPS), use=np.array(use), eos=np.array(eos),\n                        tok_len=len(tok), prompt_len=list(plen)[0], prompts=np.array(prompts, np.int32),\n                        humans=np.array(humans, np.int32), seconds=time.time() - t0, H2=H2, n=n,\n                        **{f\'g_{k}\': v for k, v in G.items()})\n    os.replace(path + \'.part.npz\', path)\n    del model; gc.collect(); torch.cuda.empty_cache()\n    notify(f\'{name}@n{n} おわり（{time.time() - t0:.0f} 秒）\')\n\n\n@torch.no_grad()\ndef self_test_a27(device):\n    """小さなランダムのモデルで：n ＝ 4 が A2-4 の block と同じトークン・同じ ℓ を出すか。n ＝ 2 で 2 トークンの写しが 0 か。"""\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    hf_id = \'hf-internal-testing/tiny-random-GPT2LMHeadModel\'\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    valid, _ = valid_mask(tok, model, device)\n    global CONT\n    keep = CONT\n    CONT = 40\n    try:\n        prompts = [[5, 6, 7, 8] * 4, [9, 10, 11, 12] * 4]\n        for T in [0.5, 1.0]:\n            g1 = torch.Generator(device=device).manual_seed(3)\n            a = generate_pen(model, prompts, T, valid, g1, 33, \'block\')\n            g2 = torch.Generator(device=device).manual_seed(3)\n            b = generate_ngram(model, prompts, T, valid, g2, 33, 4)\n            assert (a[\'toks\'] == b[\'toks\']).all(), (\'n ＝ 4 が A2-4 の block と違うトークンを出した\', T)\n            assert np.allclose(a[\'ell\'], b[\'ell\']) and (a[\'cand\'] == b[\'cand\']).all()\n            for n in (2, 3, 6):\n                g = torch.Generator(device=device).manual_seed(0)\n                r = generate_ngram(model, prompts, T, valid, g, 33, n)\n                for bb in range(2):\n                    seq = list(prompts[bb]) + list(r[\'toks\'][bb])\n                    seen = set()\n                    for i in range(n - 1, len(seq)):\n                        gram = tuple(seq[i - n + 1:i + 1])\n                        if i >= len(prompts[bb]):\n                            assert gram not in seen, (\'写しが残っている\', n, T, bb, i)\n                        seen.add(gram)\n        ci = CopyIndexN([1, 2, 3, 1, 2], 3)\n        assert set(ci.candidates()) == {3}\n        assert all(T in TEMPS for T in A7_TEMPS)\n    finally:\n        CONT = keep\n    return \'OK\'\n'
assert hashlib.sha256(SRC7C.encode()).hexdigest() == '8f8776cbe4a0e1a7b1f676c4f84ba70fb9c6c0b46ef1466969e09f0ba062b550'
exec(SRC22)
exec(SRC24)
exec(SRC7C)
print('コードを置きました')


In [ ]:
# セル3：自己テスト → 記事（A2-2 と同じか確かめる）→ n ＝ 8・6・3・2 → 保存（結果は表示しない）
import json, sys, transformers
GROUP = 'pythia'
try:
    print('自己テスト：', self_test('cuda'), self_test_pen('cuda'), self_test_a27('cuda'))
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_7', exist_ok=True)
    assert SEED == 20261027, ('記事を選ぶ種が A2-2 と違います', SEED)
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    name, hf_id = A7_GROUPS[GROUP]
    for n in A7_NS:
        run_ngram(name, hf_id, docs, '/content/drive/MyDrive/logit-thermo/A2_7', 'cuda', n, notify=lambda s: notify('✅ ' + s, tags='white_check_mark'))
    json.dump(dict(gate='docs/A2_7_GATE.md', group=GROUP, a2_2_collect_sha256='e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', a2_4_collect_sha256='6ef87866ae7d895d63c6e07d48a58a2ddf4111a4802ffe98dbe7f61892c96b57',
                   a2_7_collect_sha256='8f8776cbe4a0e1a7b1f676c4f84ba70fb9c6c0b46ef1466969e09f0ba062b550', torch=torch.__version__, transformers=transformers.__version__,
                   python=sys.version.split()[0], gpu=torch.cuda.get_device_name(0)),
              open('/content/drive/MyDrive/logit-thermo/A2_7' + f'/run_meta_{GROUP}.json', 'w'), ensure_ascii=False, indent=1)
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-7 のデータ集め（Pythia-1B）で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：A2-7 の Pythia-1B を A2_7 に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
